In [ ]:
# Public-repository path setup.
# Run from anywhere inside the repository, or set CEFTAZIDIME_PROJECT_ROOT.
import os
from pathlib import Path

def _repo_root():
    env = os.environ.get("CEFTAZIDIME_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "README.md").exists() and (candidate / "03_Notebooks").exists():
            return candidate
    return here

def _previous_project_root(project_root):
    env = os.environ.get("GENOME_MIC_AMR_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    return (project_root / "external" / "Genome_MIC_AMR_Emergence").resolve()

PROJECT_ROOT = _repo_root()

#@title Figure 2A - Ceftazidime MIC distribution in 176 blaTEM-1-only pathogens
# Purpose:
# Show the continuous MIC distribution and the empirically defined
# 16 high-MIC pathogens.
#
# High-MIC definition from Notebook 01:
# log2 MIC > Q3 + 1.5 x IQR
# Established upper fence = log2 MIC 2 (= 4 mg/L).

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
PROJECT_ROOT = _repo_root()

SAMPLE_FILE = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "10_Whole_Chromosome_Unitigs"
    / "10_unitig_sample_order.csv"
)

FIGURE_DIR = (
    PROJECT_ROOT
    / "06_Manuscript"
    / "Figures"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_PNG = (
    FIGURE_DIR
    / "Figure2A_MIC_distribution_high_MIC_pathogens.png"
)

OUTPUT_PDF = (
    FIGURE_DIR
    / "Figure2A_MIC_distribution_high_MIC_pathogens.pdf"
)

UPPER_FENCE = 2.0

samples = (
    pd.read_csv(
        SAMPLE_FILE
    )
    .sort_values(
        "sample_index"
    )
    .reset_index(
        drop=True
    )
)

assert len(
    samples
) == 176

assert "log2_mic" in samples.columns

y = samples[
    "log2_mic"
].to_numpy(
    dtype=float
)

high_mask = (
    y > UPPER_FENCE
)

assert high_mask.sum() == 16

# Order all pathogens by MIC.
order = np.argsort(
    y,
    kind="stable",
)

ordered_y = y[
    order
]

ordered_high = high_mask[
    order
]

x = np.arange(
    1,
    len(
        ordered_y
    ) + 1,
)

fig, ax = plt.subplots(
    figsize=(8.5, 5.2)
)

# 160 remaining blaTEM-1-only pathogens.
ax.scatter(
    x[
        ~ordered_high
    ],
    ordered_y[
        ~ordered_high
    ],
    s=24,
    facecolors="white",
    edgecolors="black",
    linewidths=0.7,
    label="160 remaining pathogens",
)

# 16 high-MIC pathogens.
ax.scatter(
    x[
        ordered_high
    ],
    ordered_y[
        ordered_high
    ],
    s=34,
    facecolors="black",
    edgecolors="black",
    linewidths=0.7,
    label="16 high-MIC pathogens",
)

# Established upper-outlier fence.
ax.axhline(
    UPPER_FENCE,
    linestyle="--",
    linewidth=1.0,
)

ax.text(
    2,
    UPPER_FENCE + 0.18,
    "Upper-outlier fence = 2",
    fontsize=9,
    va="bottom",
)

ax.set_xlabel(
    "176 blaTEM-1-only pathogens ordered by MIC"
)

ax.set_ylabel(
    "Ceftazidime log₂(MIC)"
)


ax.spines[
    "top"
].set_visible(
    False
)

ax.spines[
    "right"
].set_visible(
    False
)

fig.tight_layout()

fig.savefig(
    OUTPUT_PNG,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    OUTPUT_PDF,
    bbox_inches="tight",
)

plt.show()
plt.close(
    fig
)

print(
    "Figure 2A created."
)

print(
    "High-MIC pathogens:",
    int(
        high_mask.sum()
    ),
)

print(
    OUTPUT_PNG
)

In [ ]:
#@title Figure 2B - Chromosomal backgrounds of the 16 high-MIC pathogens
# Purpose:
# Visualize genome-wide chromosomal SNP relatedness among the 176
# blaTEM-1-only pathogens.
#
# The plot uses kernel PCA of the previously established SNP-relatedness
# matrix K. The same 16 high-MIC pathogens from Figure 2A are highlighted.
#
# This is a visualization of chromosomal background structure.
# The formal relatedness test was performed previously in Notebook 01.

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
OLD_PROJECT_ROOT = _previous_project_root(PROJECT_ROOT)

SAMPLE_FILE = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "10_Whole_Chromosome_Unitigs"
    / "10_unitig_sample_order.csv"
)

K_FILE = (
    OLD_PROJECT_ROOT
    / "04_Population_Structure"
    / "Notebook04"
    / "04_genome_wide_relatedness_matrix.npz"
)

INDEX_FILE = (
    OLD_PROJECT_ROOT
    / "02_Data_Preparation"
    / "Notebook03"
    / "03_ceftazidime_pathogen_index.csv"
)

FIGURE_DIR = (
    PROJECT_ROOT
    / "06_Manuscript"
    / "Figures"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_PNG = (
    FIGURE_DIR
    / "Figure2B_high_MIC_chromosomal_backgrounds.png"
)

OUTPUT_PDF = (
    FIGURE_DIR
    / "Figure2B_high_MIC_chromosomal_backgrounds.pdf"
)

UPPER_FENCE = 2.0

for path in [
    SAMPLE_FILE,
    K_FILE,
    INDEX_FILE,
]:
    assert path.exists(), (
        f"Required input not found: {path}"
    )

# ------------------------------------------------------------
# Load the fixed 176-pathogen cohort
# ------------------------------------------------------------

samples = (
    pd.read_csv(
        SAMPLE_FILE
    )
    .sort_values(
        "sample_index"
    )
    .reset_index(
        drop=True
    )
)

assert len(
    samples
) == 176

assert "biosample" in samples.columns
assert "log2_mic" in samples.columns

y = samples[
    "log2_mic"
].to_numpy(
    dtype=float
)

high_mask = (
    y > UPPER_FENCE
)

assert high_mask.sum() == 16

# ------------------------------------------------------------
# Load the previous genome-wide relatedness matrix
# ------------------------------------------------------------

with np.load(
    K_FILE
) as archive:

    square_keys = []

    for key in archive.files:

        value = archive[
            key
        ]

        if (
            value.ndim == 2
            and value.shape[0] == value.shape[1]
        ):
            square_keys.append(
                key
            )

    assert len(
        square_keys
    ) == 1, (
        "Expected exactly one square matrix in the relatedness NPZ.\n"
        f"Square-matrix keys found: {square_keys}\n"
        f"All keys: {archive.files}"
    )

    K_full = np.asarray(
        archive[
            square_keys[0]
        ],
        dtype=float,
    )

print(
    "Relatedness matrix key:",
    square_keys[0],
)

print(
    "Full relatedness matrix shape:",
    K_full.shape,
)

# ------------------------------------------------------------
# Map the 176 BioSamples to the previous K matrix
# ------------------------------------------------------------

index_table = pd.read_csv(
    INDEX_FILE
)

assert len(
    index_table
) == K_full.shape[0], (
    "Pathogen-index row count does not match the relatedness matrix."
)

target_biosamples = (
    samples[
        "biosample"
    ]
    .astype(
        str
    )
    .str.strip()
)

# Identify the BioSample column by content rather than guessing its name.
matching_identifier_columns = []

for column in index_table.columns:

    values = (
        index_table[
            column
        ]
        .astype(
            str
        )
        .str.strip()
    )

    if values.duplicated().any():
        continue

    if target_biosamples.isin(
        set(
            values
        )
    ).all():

        matching_identifier_columns.append(
            column
        )

assert len(
    matching_identifier_columns
) == 1, (
    "Could not uniquely identify the BioSample column in the "
    "previous pathogen-index file.\n"
    f"Matching columns: {matching_identifier_columns}\n"
    f"Available columns: {list(index_table.columns)}"
)

BIOSAMPLE_COLUMN = (
    matching_identifier_columns[0]
)

print(
    "BioSample column in previous index:",
    BIOSAMPLE_COLUMN,
)

previous_biosamples = (
    index_table[
        BIOSAMPLE_COLUMN
    ]
    .astype(
        str
    )
    .str.strip()
)

biosample_to_K_row = {
    biosample: row_index
    for row_index, biosample in enumerate(
        previous_biosamples
    )
}

K_rows = np.asarray(
    [
        biosample_to_K_row[
            biosample
        ]
        for biosample in target_biosamples
    ],
    dtype=int,
)

assert len(
    np.unique(
        K_rows
    )
) == 176

K_176 = K_full[
    np.ix_(
        K_rows,
        K_rows,
    )
]

assert K_176.shape == (
    176,
    176,
)

assert np.allclose(
    K_176,
    K_176.T,
    atol=1e-10,
)

# ------------------------------------------------------------
# Kernel PCA of the 176 x 176 SNP-relatedness matrix
# ------------------------------------------------------------

n = K_176.shape[
    0
]

H = (
    np.eye(
        n
    )
    - np.ones(
        (
            n,
            n,
        )
    )
    / n
)

K_centered = (
    H
    @ K_176
    @ H
)

K_centered = (
    K_centered
    + K_centered.T
) / 2.0

eigenvalues, eigenvectors = np.linalg.eigh(
    K_centered
)

order = np.argsort(
    eigenvalues
)[::-1]

eigenvalues = eigenvalues[
    order
]

eigenvectors = eigenvectors[
    :,
    order
]

positive = (
    eigenvalues > 1e-10
)

positive_eigenvalues = eigenvalues[
    positive
]

positive_eigenvectors = eigenvectors[
    :,
    positive
]

assert len(
    positive_eigenvalues
) >= 2

coordinates = (
    positive_eigenvectors[
        :,
        :2
    ]
    * np.sqrt(
        positive_eigenvalues[
            :2
        ]
    )
)

positive_trace = float(
    positive_eigenvalues.sum()
)

pc1_fraction = (
    positive_eigenvalues[
        0
    ]
    / positive_trace
)

pc2_fraction = (
    positive_eigenvalues[
        1
    ]
    / positive_trace
)

# ------------------------------------------------------------
# Label the 16 high-MIC pathogens H01-H16
# Highest MIC first, then sample index.
# ------------------------------------------------------------

high_indices = np.flatnonzero(
    high_mask
)

high_order = sorted(
    high_indices,
    key=lambda i: (
        -y[i],
        i,
    ),
)

high_labels = {
    sample_index: f"H{rank:02d}"
    for rank, sample_index in enumerate(
        high_order,
        start=1,
    )
}

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(7.2, 6.4)
)

# 160 remaining blaTEM-1-only pathogens.
ax.scatter(
    coordinates[
        ~high_mask,
        0
    ],
    coordinates[
        ~high_mask,
        1
    ],
    s=28,
    facecolors="white",
    edgecolors="black",
    linewidths=0.6,
    alpha=0.75,
    label="160 remaining pathogens",
)

# 16 high-MIC pathogens.
ax.scatter(
    coordinates[
        high_mask,
        0
    ],
    coordinates[
        high_mask,
        1
    ],
    s=52,
    facecolors="black",
    edgecolors="black",
    linewidths=0.7,
    label="16 high-MIC pathogens",
)

ax.set_xlabel(
    f"Kernel PC1 ({100 * pc1_fraction:.1f}% of positive trace)"
)

ax.set_ylabel(
    f"Kernel PC2 ({100 * pc2_fraction:.1f}% of positive trace)"
)

ax.set_title(
    "Genome-wide chromosomal SNP relatedness"
)


ax.spines[
    "top"
].set_visible(
    False
)

ax.spines[
    "right"
].set_visible(
    False
)

fig.tight_layout()

fig.savefig(
    OUTPUT_PNG,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    OUTPUT_PDF,
    bbox_inches="tight",
)

plt.show()
plt.close(
    fig
)

print(
    "Figure 2B created."
)

print(
    "16 high-MIC pathogens highlighted:",
    int(
        high_mask.sum()
    ),
)

print(
    OUTPUT_PNG
)

In [ ]:
#@title Final combined Figure 2 - MIC distribution and chromosomal similarity
# Purpose:
# Combine Figure 2A and Figure 2B into one publication figure.
#
# Panel A:
# Ceftazidime log2(MIC) distribution in the 176 blaTEM-1-only pathogens.
#
# Panel B:
# Kernel-PCA representation of the SNP-based chromosomal similarity matrix K.
#
# Symbol definitions are intentionally omitted from the panels and described
# in the manuscript figure caption:
#   open circles  = 160 remaining pathogens
#   filled circles = 16 high-MIC pathogens

import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Verify that the two preceding Figure 2 cells have been run
# ------------------------------------------------------------

assert len(ordered_y) == 176
assert len(ordered_high) == 176
assert int(np.sum(ordered_high)) == 16

assert coordinates.shape == (176, 2)
assert len(high_mask) == 176
assert int(np.sum(high_mask)) == 16

# ------------------------------------------------------------
# Final combined output paths
# ------------------------------------------------------------

FIG2_COMBINED_PNG = (
    FIGURE_DIR
    / "Figure2_combined_MIC_distribution_and_chromosomal_similarity.png"
)

FIG2_COMBINED_PDF = (
    FIGURE_DIR
    / "Figure2_combined_MIC_distribution_and_chromosomal_similarity.pdf"
)

# ------------------------------------------------------------
# Draw combined Figure 2
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14.5, 5.8),
    gridspec_kw={
        "width_ratios": [1.18, 1.0],
        "wspace": 0.25,
    },
)

ax_a, ax_b = axes

# ------------------------------------------------------------
# Panel A - MIC distribution
# ------------------------------------------------------------

ax_a.scatter(
    x[~ordered_high],
    ordered_y[~ordered_high],
    s=24,
    facecolors="white",
    edgecolors="black",
    linewidths=0.7,
)

ax_a.scatter(
    x[ordered_high],
    ordered_y[ordered_high],
    s=34,
    facecolors="black",
    edgecolors="black",
    linewidths=0.7,
)

ax_a.axhline(
    UPPER_FENCE,
    linestyle="--",
    linewidth=1.0,
)

ax_a.text(
    2,
    UPPER_FENCE + 0.18,
    "Upper-outlier fence = 2",
    fontsize=9,
    va="bottom",
)

ax_a.set_xlabel(
    "176 blaTEM-1-only pathogens ordered by MIC"
)

ax_a.set_ylabel(
    "Ceftazidime log₂(MIC)"
)

ax_a.spines["top"].set_visible(False)
ax_a.spines["right"].set_visible(False)

# ------------------------------------------------------------
# Panel B - chromosomal similarity
# ------------------------------------------------------------

ax_b.scatter(
    coordinates[~high_mask, 0],
    coordinates[~high_mask, 1],
    s=28,
    facecolors="white",
    edgecolors="black",
    linewidths=0.6,
    alpha=0.75,
)

ax_b.scatter(
    coordinates[high_mask, 0],
    coordinates[high_mask, 1],
    s=52,
    facecolors="black",
    edgecolors="black",
    linewidths=0.7,
)

ax_b.set_xlabel(
    f"Kernel PC1 ({100 * pc1_fraction:.1f}% of positive trace)"
)

ax_b.set_ylabel(
    f"Kernel PC2 ({100 * pc2_fraction:.1f}% of positive trace)"
)

ax_b.set_title(
    "Genome-wide chromosomal SNP relatedness"
)

ax_b.spines["top"].set_visible(False)
ax_b.spines["right"].set_visible(False)

# ------------------------------------------------------------
# Panel labels
# ------------------------------------------------------------

ax_a.text(
    -0.10,
    1.03,
    "A",
    transform=ax_a.transAxes,
    fontsize=16,
    fontweight="bold",
    va="top",
    ha="left",
)

ax_b.text(
    -0.10,
    1.03,
    "B",
    transform=ax_b.transAxes,
    fontsize=16,
    fontweight="bold",
    va="top",
    ha="left",
)

# ------------------------------------------------------------
# Save publication Figure 2
# ------------------------------------------------------------

fig.savefig(
    FIG2_COMBINED_PNG,
    dpi=600,
    bbox_inches="tight",
)

fig.savefig(
    FIG2_COMBINED_PDF,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print("Final combined Figure 2 created.")
print("Legends removed from the panels; symbol definitions belong in the caption.")
print(FIG2_COMBINED_PNG)
print(FIG2_COMBINED_PDF)
